# Bike Rental Prediction

여러분이 도시에 있는 <b>공유 자전거 서비스 운영자</b>라고 생각해봅시다.<br>매일 많은 사람들이 출퇴근, 운동, 여가를 위해 자전거를 빌려 탑승합니다.

그럼 이런 질문이 자연스럽게 떠오릅니다:

> “날씨, 계절, 시간대 등의 여러 정보를 활용해서 앞으로 <b>몇 대의 자전거가 대여될지 미리 예측</b>할 수 있을까?”

이번 실습에서는 머신러닝을 활용하여 <b>자전거 수요 예측</b> 모델을 개발해봅니다.

실제로 많은 공유 자전거 회사나 킥보드/카셰어링 서비스에서도 이런 모델을 활용해 얼마나 많은 자전거나 차량을 준비해야 하는지 결 정합니다.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("data/hour.csv")

df["timestamp"] = pd.to_datetime(df["dteday"]) + pd.to_timedelta(df["hr"], unit="h")
df = df.drop(columns = ["instant", "dteday", "casual", "registered"])
df = df.sort_values("timestamp").reset_index(drop=True)

print(df.head())
print("\nDataframe shape: ", df.shape)

## 데이터 설명

데이터는 `pandas.DataFrame`으로 주어지며, 각 행(row)은 한 시간 단위의 기록을 의미합니다.

<b>주요 컬럼 설명</b>

- timestamp : 날짜와 시각 (DataFrame is sorted by timestamp)
- season : 계절 (1:봄, 2:여름, 3:가을, 4:겨울)
- yr : 년도 (0: 2011년, 1:2012년)
- mnth : 월 (1 ~ 12)
- hr : 시각 (0 ~ 23)
- holiday : 공휴일 여부
- weekday : 요일 (0: 일요일, ..., 6: 토요일)
- workingday : 평일 여부 (평일 = 1, 주말/공휴일 = 0)
- weathersit : 날씨 상태
    - 1: 맑음, 구름 조금
    - 2: 안개, 흐림
    - 3: 가벼운 눈, 약한 비
    - 4: 폭우, 폭설, 천둥번개 + 안개
- temp : 기온 (섭씨 온도를 최대 값 41로 나누어 0~1사이의 값으로 normalize)
- atemp: 체감 기온 (최대값 50으로 나누어 normalize)
- hum: 습도 (최대값 100으로 나누어 normalize)
- windspeed: 풍속 (최대값 67로 나누어 normalize)
- cnt: 자전거 대여 총 대수

In [ ]:
CATEGORICAL_COLS = ["season", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit", "yr"]
NUMERIC_COLS = ["temp", "atemp", "hum", "windspeed"]

for col in CATEGORICAL_COLS:
    df[col] = df[col].astype("category")

y = df["cnt"].astype(float)
X = df.drop(columns=["cnt", "timestamp"])

<mark>실습</mark>

 - 머신러닝의 각 과정에 해당하는 코드들을 모두 작성하세요:
   - EDA (Exporatory Data Analaysis)
   - 데이터 전처리
   - feature engineering
   - 모델 학습 (training)
   - 평가 (evaluation)

- 성능 개선을 위해 <b>최소 3번 이상의 실험</b>을 수행하세요
   - 예: 모델 종류 변경, 하이퍼파라미터 조정, feature engineering 등
   - <mark>주의</mark>: 각 실험에 대한 <b>코드와 결과를 모두 기록</b>해 두세요. (즉, 결과 재현이 가능해야함).

- 최종적으로 예측 모델은 다음과 같은 성능을 만족해야 합니다:
   - $\text{RMSE} < 125$
   - $R^2 > 0.68$

In [ ]:
##### WRITE YOUR CODE #####

## Visualize prediction result

아래 제공된 코드를 활용하여 예측 결과를 시각화 해보세요.

<mark>실습</mark> 예측모델의 성능이 떨어지는 이유는 무엇이고, 어떻게 하면 성능을 더 개선할 수 고민해보세요.

In [ ]:
def plot_predictions(y_true, y_pred, save_path = None):
    plt.figure(figsize=(7,7))
    plt.scatter(y_true, y_pred, marker='X', label="Predictions")

    # Reference diagonal (perfect predictions)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], color="black", linestyle="--", label="Ideal")
   
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.axis("equal")  # keep x=y scaling
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        
    plt.show()

In [ ]:
plot_predictions(y_test, y_pred)

In [ ]:
def plot_predictions_over_index(y_true, y_pred, downsample_points=500, save_path = None):
    if len(y_true) > downsample_points: # Downsample if too many points
        x_axis = np.linspace(0, len(y_true) - 1, downsample_points, dtype=int)
        y_true = np.array(y_true)[x_axis]
        y_pred = np.array(y_pred)[x_axis]
    else:
        x_axis = range(len(y_true))

    plt.figure(figsize=(12,5))
    plt.plot(x_axis, y_true, label="Actual", linewidth=1.5)
    plt.plot(x_axis, y_pred, label="Predicted", linewidth=1.5, alpha=0.8)
    plt.title("Bike Rentals")
    plt.xlabel("Index (time)")
    plt.ylabel("Rental count")
    plt.legend()
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150)

    plt.show()

In [ ]:
plot_predictions_over_index(y_test, y_pred)